# **Mapping from Sentinel-1 and Sentinel-2 data**

Francescopaolo Sica


### **Description:**
This hands-on tutorial introduces participants to the synergistic use of Sentinel-1 (SAR) and Sentinel-2 (optical) satellite data for Earth observation applications, with a focus on flood mapping. Participants will learn how to access, pre-process, and combine data from both sensors to create reliable flood extent maps. The tutorial will cover data fusion techniques, basic machine learning models, and interpretation of results. Designed for researchers and practitioners in remote sensing, the session emphasizes practical workflows using open-source tools and publicly available datasets.

In [1]:
# Step 1: Install dependencies and initialize GEE
!pip install -q geemap

import ee
import geemap
import matplotlib.pyplot as plt

PROJECT = "igarss26"

try:
    ee.Initialize(project=PROJECT)
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

/dmidata/users/cgf/miniforge3/envs/landfastice/lib/python3.10/site-packages/google/api_core/_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)



Successfully saved authorization token.


In [2]:
# Connect your drive
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Define export function for saving results

def export_to_drive(image, region, filename, folder='EarthEngine', scale=10):
    task = ee.batch.Export.image.toDrive(
        image=image.clip(region),
        description=filename,
        folder=folder,
        fileNamePrefix=filename,
        region=region,
        scale=scale,
        maxPixels=1e10
    )
    task.start()
    print(f"Started export task: {filename}")
    return task


In [ ]:
# Step 2: Define flood region and dates (Bangladesh, 2020 flood)

# Define your AOI and load Sentinel-1, Sentinel-2
min_lon, min_lat, max_lon, max_lat = [89.5, 23.5, 90.5, 24.5]
region = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat]) #region for download
region_large = ee.Geometry.Rectangle([min_lon-1, min_lat-1, max_lon+1, max_lat+1]) # region for visualizing and saving results

# Sentinel-1 before/after with median composites
before = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(region) \
    .filterDate("2020-06-01", "2020-06-30") \
    .filter(ee.Filter.eq("instrumentMode", "IW")) \
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV")) \
    .median()

after = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filterBounds(region) \
    .filterDate("2020-07-20", "2020-07-30") \
    .filter(ee.Filter.eq("instrumentMode", "IW")) \
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV")) \
    .median()


# Load Sentinel-2 RGB image
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
    .filterBounds(region) \
    .filterDate("2020-07-20", "2020-07-31")

if collection.size().getInfo() == 0:
    raise ValueError("No Sentinel-2 image found for the given region and date range.")

s2 = collection.median()

if s2 is None:
    raise ValueError("No Sentinel-2 image found for the given region and date range.")

count = collection.size().getInfo()

print("Number of images found:", count)


try:
    band_names = s2.bandNames().getInfo()
    print("Sentinel-2 bands:", band_names)
except Exception as e:
    print("Error: Image may be empty or invalid.", e)

# Load GlobalSurfaceWater
water = ee.Image("JRC/GSW1_3/GlobalSurfaceWater").select("occurrence")

# Load DEM from SRTM
srtm = ee.Image("USGS/SRTMGL1_003")
srtm = srtm.clip(region_large)

Number of images found: 24
Sentinel-2 bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


/usr/local/lib/python3.11/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for JRC/GSW1_3/GlobalSurfaceWater! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/JRC_GSW1_3_GlobalSurfaceWater

  warnings.warn(warning, category=DeprecationWarning)


In [ ]:
# Prepare color palettes for visualization:

rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

sar_vis = {
    "min": -25,
    "max": 0,
    'palette': ['black', 'white']
}

diff_vis = {
    "min": -5,
    "max": 5,
    "palette": ["red", "white", "green"]
}

flood_vis = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'blue']
}

srtm_vis = {
    'min': 0,
    'max': 1000,
    'palette': ['0000ff', '00ffff', 'ffff00', 'ff0000', 'ffffff']
}


In [ ]:
# Step 3: Compute difference image
diff_db = before.subtract(after)


In [ ]:
# Optional: compute and plot histogram

# hist_dict = diff_db.reduceRegion(
#     reducer=ee.Reducer.histogram(maxBuckets=100),
#     geometry=region,
#     scale=10,
#     maxPixels=1e9
# )

# # Get the band name
# band = diff_db.bandNames().get(0).getInfo()

# # Get histogram dictionary
# hist = hist_dict.get(band).getInfo()
# counts = hist['histogram']
# bins = hist['bucketMeans']

# # Plot the histogram
# plt.figure(figsize=(8, 5))
# plt.bar(bins, counts, width=0.5)
# plt.title('Histogram of SAR Backscatter Difference (dB)')
# plt.xlabel('Backscatter Difference (dB)')
# plt.ylabel('Pixel Count')
# plt.grid(True)
# plt.show()

In [ ]:
# Step 4: compute flood map by setting a threshold

threshold_value_dB = 4

# Apply threshold
flood_th = diff_db.gt(threshold_value_dB)

# Filter out permanent water bodies
permanent_water = water.gt(90)  # only mask areas with >90% occurrence
flood_th = flood_th.updateMask(permanent_water.Not())

# Filetr out high slope terrains
slope = ee.Terrain.slope(srtm)
flood = flood_th.updateMask(slope.lt(5))  # only areas flatter than 5°

In [ ]:
# Step 5: Display map
Map = geemap.Map(center=[24.0, 90.5], zoom=9)
Map.addLayer(srtm, srtm_vis, 'SRTM DEM')
Map.addLayer(s2, rgb_vis, "Sentinel-2 RGB")
Map.addLayer(before.select("VV"), sar_vis, "S1 VV Before Flood")
Map.addLayer(after.select("VV"), sar_vis, "S1 VV After Flood")
Map.addLayer(diff_db.select("VV"), diff_vis, "S1 VV Difference")
Map.addLayer(flood.select(0).updateMask(flood.select(0)), flood_vis, "Flood Mask")
Map.addLayerControl()
Map

Map(center=[24.0, 90.5], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(…

In [ ]:
# Export data of interest

# export_to_drive(flood, region_large, 'Bangladesh_2020_FloodMap_lowres', scale=100)
# export_to_drive(before.select("VV"), region_large, 'Bangladesh_2020_Flood_SAR_before_lowres', scale=100)
# export_to_drive(after.select("VV"), region_large, 'Bangladesh_2020_Flood_SAR_after_lowres', scale=100)